# Disaster damage demo: can a written claim sway a vision model?

This portable notebook replays a recorded Qwen3-VL run on 12 xBD buildings. It includes the recorded predictions and one before/after image pair, so **Run All works even when this `.ipynb` file is opened outside the repository**. The model is not run by default. The final section shows how to rerun inference with the project code and source data.

The four prompt conditions for each building are images only, a matching written claim, and two conflicting claims. The images and classification question stay fixed within a building. Source imagery and labels: [xBD / xView2](https://xview2.org/dataset).

## Requirements

Open this file in Jupyter with a Python 3 kernel and run the cells in order. The replay uses only Python's standard library and IPython, which Jupyter provides. It needs no repository checkout, dataset download, or model weights. The embedded snapshot is a **record of a completed run**, not a new inference run.

In [ ]:
import base64
import hashlib
import json
import zlib
from IPython.display import Image, Markdown, display

# Recorded local run. This compressed snapshot contains 12 labels, 48 scored
# prompts, run metadata, and one xBD before/after crop pair.
_SNAPSHOT_B64 = (
    "..."
    "Htn/GNb/68P6v//X//p/AoA0Dy4="
)
snapshot = json.loads(zlib.decompress(base64.b64decode(_SNAPSHOT_B64)))
assert snapshot["run"]["complete"]
assert snapshot["run"]["manifest_sha256"] == snapshot["manifest_sha256"]
assert len(snapshot["buildings"]) == 12 and len(snapshot["rows"]) == 48
print(f"Loaded recorded run: {snapshot['run']['model_id']}")
print(f"Model revision: {snapshot['run']['model_revision']}")
print(f"Buildings: {len(snapshot['buildings'])}; prompts: {len(snapshot['rows'])}")

## 1. Inspect one image pair

The yellow box identifies the same target building in the before and after images. It is a location marker, not a damage label.

In [ ]:
example = snapshot["example"]
print(f"Building: {example['building_id']}")
print(f"xBD label: {example['true_label']}")
display(Markdown("**Before disaster**"))
display(Image(data=base64.b64decode(example["pre_png"])))
display(Markdown("**After disaster**"))
display(Image(data=base64.b64decode(example["post_png"])))

## 2. Inspect the fixed question and four claims

The prompts below were saved from the project’s `demo_conditions` function when this snapshot was made. The model chooses A, B, C, or D for each prompt.

In [ ]:
for condition in example["conditions"]:
    print(f"\n[{condition['name']}; claim={condition['text_claim']!r}; gap={condition['conflict_distance']}]")
    print(condition["prompt"])

## 3. Summarize the 12-building run

The scorer took Qwen’s next-token logits for A/B/C/D and applied softmax over those four choices. Each row’s shares sum to 1; they are **relative choice scores**, not calibrated probabilities of being correct.

In [ ]:
rows = snapshot["rows"]
baseline = [row for row in rows if row["condition"] == "image_only"]
conflicts = [row for row in rows if row["condition"] == "conflicting_text"]
eligible = [row for row in conflicts if row["image_only_prediction"] != row["text_claim"]]
correct = sum(row["predicted_label"] == row["true_label"] for row in baseline)
followed = sum(row["predicted_label"] == row["text_claim"] for row in conflicts)
flipped = sum(row["changed_toward_text"] for row in eligible)
assert len(baseline) == 12 and len(conflicts) == 24
assert all(abs(sum(row["choice_probabilities"].values()) - 1) < 1e-8 for row in rows)
display(Markdown(
    f"**Image-only agreement with xBD:** {correct}/{len(baseline)}  \n"
    f"**Conflicting claims followed:** {followed}/{len(conflicts)}  \n"
    f"**Eligible conflicts that moved toward the claim:** {flipped}/{len(eligible)}"
))
print("Eligible means the claim differs from the image-only prediction.")

## 4. Compare the same building across conditions

“Share” is the winning choice’s softmax share within that prompt. “Changed?” compares the answer with the image-only answer.

In [ ]:
example_rows = [row for row in rows if row["building_id"] == example["building_id"]]
assert len(example_rows) == 4
lines = [
    "| Condition | Claim | Predicted answer | Share | Changed? |",
    "|---|---|---|---:|---|",
]
for row in example_rows:
    claim = row["text_claim"] or "—"
    changed = "—" if row["prediction_changed"] is None else (
        "Yes" if row["prediction_changed"] else "No"
    )
    lines.append(
        f"| {row['condition']} | {claim} | {row['predicted_choice']}: "
        f"{row['predicted_label']} | {row['confidence']:.3f} | {changed} |"
    )
display(Markdown("\n".join(lines)))
image_only = next(row for row in example_rows if row["condition"] == "image_only")
print("A/B/C/D shares for the image-only prompt:")
for choice, label in zip("ABCD", ("no damage", "minor damage", "major damage", "destroyed")):
    print(f"  {choice} ({label}): {image_only['choice_probabilities'][choice]:.6f}")

## Interpretation

In this recorded sample, 6 of 12 image-only predictions matched xBD. Qwen chose the claimed class in all 24 conflicting prompts. In 22 of those, the claim differed from the image-only prediction and the answer moved toward the claim; the other two already agreed with the claim. This small selected sample demonstrates the pipeline and text sensitivity. It does not establish a general accuracy rate or a causal relationship between confidence and flips.

## 5. Optional: rerun the model with the repository

The replay above is fully portable. A **new inference run** still needs this repository, xBD crops, the `demo` dependencies, and an accelerator. On a repository checkout, follow [the runbook](../docs/midterm-demo.md) or use the cell below. Set `REPO_ROOT` to the checkout path if Jupyter started elsewhere; set `RUN_INFERENCE = True` only when you want to load and run Qwen. The first run downloads model weights and the 48-prompt run may take substantial time.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_ROOT = None  # Example: Path("/path/to/345-Disaster-Model")
RUN_INFERENCE = False
OUTPUT_DIR = "results/notebook-run"

if RUN_INFERENCE:
    candidates = []
    if REPO_ROOT is not None:
        candidates.append(Path(REPO_ROOT))
    if os.environ.get("DISASTER_MODEL_ROOT"):
        candidates.append(Path(os.environ["DISASTER_MODEL_ROOT"]))
    candidates.extend((Path.cwd(), *Path.cwd().parents))
    root = next(
        (path.resolve() for path in candidates
         if (path / "scripts/run_demo.py").is_file()
         and (path / "data/processed/manifest.jsonl").is_file()),
        None,
    )
    if root is None:
        raise RuntimeError("Set REPO_ROOT to a checkout with prepared xBD crops.")
    common = [
        sys.executable, str(root / "scripts/run_demo.py"),
        "--manifest", str(root / "data/processed/manifest.jsonl"),
        "--output", str(root / OUTPUT_DIR),
    ]
    subprocess.run([*common, "--pilot-only"], cwd=root, check=True)
    subprocess.run(common, cwd=root, check=True)
else:
    print("Portable replay complete. Set RUN_INFERENCE = True for a new Qwen run.")